<a href="https://colab.research.google.com/github/shayanR10/FIV1/blob/main/FIV1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# README: This is my dinosaur identifier AI model, built in PyTorch via Google Colab; pretty self explanatory,
# so I'll just cut to the chase and show you how the whole thing works:
# (also make sure to run these cells in the order in which they appear in this notebook).

In [ ]:
#prerequisites, run this cell first; make sure to also create a Google Kaggle Account along with Colab secrets named "KAGGLE_USERNAME" , "KAGGLE_SLUG" , and "KAGGLE_KEY" for your Google Kaggle
# username, dataset slug, and custom API key, respectively.

!pip install ftfy regex tqdm -q
!pip install git+https://github.com/openai/CLIP.git -q
!pip install kagglehub -q

In [ ]:
#Kaggle connection setup; data pipeline

import os
from google.colab import userdata
import kagglehub

username = userdata.get('KAGGLE_USERNAME')
slug = userdata.get('KAGGLE_SLUG')
key = userdata.get('KAGGLE_KEY')
os.environ["KAGGLE_USERNAME"] = username
os.environ["KAGGLE_KEY"] = key

print("Authenticating")

try:
    path = kagglehub.dataset_download(f"{username}/{slug}")
    print("Success")
    print(path)
except Exception as e:
    print("Failed.")
    kagglehub.login()

In [ ]:
# Image acquisition, filtering, checkpointing, and upload

import os
import time
import json
import shutil
from urllib.parse import urlparse
import requests
import torch
import clip
from google.colab import userdata
from PIL import Image
import kagglehub

# setup
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
KAGGLE_USERNAME = userdata.get('KAGGLE_USERNAME')
KAGGLE_SLUG = userdata.get('KAGGLE_SLUG')
datadirectory = "./fossil-data"

# Open-ended dynamic list: You can add any dinosaur here freely
specieslist = [
    "Tyrannosaurus Rex",
    "Triceratops Horridus",
    "Stegosaurus",
    "Velociraptor Mongoliensis",
    "Spinosaurus Aegypticus"
]
mIMperSP = 500

os.makedirs(datadirectory, exist_ok=True)

# clip loading
print("loading CLIP")
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)
print("CLIP loaded successfully.")

# metadata junk keywords
junk = [
    'toy', 'plastic', 'figure', 'game', 'render', '3d', 'plush',
    'sculpture', 'lego', 'meme', 'origami', 'suit', 'costume',
    'animatronic', 'illustration', 'drawing', 'painting', 'cartoon',
    'diorama', 'taxidermy', 'human', 'tourist', 'graffiti', 'logo'
]

# scraping pipeline with checkpointing & pagination offset
def scraping(species, max_results):
    folderpath = os.path.join(datadirectory, species.replace(" ", "_"))
    os.makedirs(folderpath, exist_ok=True)

    # Checkpoint check: if we already have files downloaded here, check count
    existing_files = [f for f in os.listdir(folderpath) if f.endswith('.jpg')]
    if len(existing_files) >= max_results:
        print(f"\n[{species}] Checkpoint found: {len(existing_files)} images already exist. Skipping download.")
        return

    print(f"\n[{species}] scraping in progress")

    url = "https://commons.wikimedia.org/w/api.php"
    wikiheaders = {
        "User-Agent": f"dinodatafinder/1.0 (https://kaggle.com/{KAGGLE_USERNAME}) Python-requests",
        "Referer": "https://commons.wikimedia.org/"
    }

    genus = species.split()[0]
    search_query = f"{genus} skeleton -toy -3d -model -game -sculpture"

    session = requests.Session()
    session.headers.update(wikiheaders)

    saved = len(existing_files)
    skippedcount = 0
    sroffset = 0

    try:
        while saved < max_results and sroffset < 200:
            searchparams = {
                "action": "query",
                "generator": "search",
                "gsrsearch": search_query,
                "gsrnamespace": "6",
                "gsrlimit": min(50, max_results - saved),
                "gsroffset": sroffset,
                "prop": "imageinfo",
                "iiprop": "url|extmetadata",
                "format": "json"
            }

            response = session.get(url, params=searchparams, timeout=10).json()

            if "query" in response and "pages" in response["query"]:
                pages = response["query"]["pages"]

                for pageid, pageinfo in pages.items():
                    if saved >= max_results:
                        break
                    if "imageinfo" in pageinfo:
                        info = pageinfo['imageinfo'][0]
                        imgurl = info['url']
                        extrameta = info.get('extmetadata', {})

                        pathparsed = urlparse(imgurl).path.lower()
                        if not any(pathparsed.endswith(ext) for ext in ['.jpg', '.jpeg', '.png', '.webp']):
                            continue

                        metadata = ""
                        if 'ImageDescription' in extrameta:
                            metadata += str(extrameta['ImageDescription'].get('value', '')).lower()
                        if 'Categories' in extrameta:
                            metadata += str(extrameta['Categories'].get('value', '')).lower()

                        if any(junkW in metadata for junkW in junk):
                            skippedcount += 1
                            continue

                        try:
                            time.sleep(0.3)
                            IMGresponse = session.get(imgurl, timeout=(5, 10))
                            if IMGresponse.status_code == 200:
                                filepath = os.path.join(folderpath, f"{saved:03d}.jpg")
                                # Skip if file somehow already exists
                                if not os.path.exists(filepath):
                                    with open(filepath, "wb") as file:
                                        file.write(IMGresponse.content)
                                    saved += 1
                        except Exception:
                            pass

                # Advance pagination offset to fetch deeper results instead of repeating top matches
                sroffset += 50
            else:
                break

        print(f"[{species}] Total valid images: {saved}. Skipped {skippedcount} via metadata.")

    except Exception as e:
        print(f"An error occurred during scraping {species}: {e}")

# filtering with strict single-skeleton enforcement
def filter_images(species):
    folderpath = os.path.join(datadirectory, species.replace(" ", "_"))
    if not os.path.exists(folderpath):
        return

    print(f"[{species}] Filtering images...")
    broken = 0
    irrelevant = 0

    # Index 0 is our strict ideal target. Indices 1-4 are rejection classes.
    comparisonprompts = [
        f"A clear, isolated photograph of a single complete museum dinosaur skeleton of a {species}",
        "A scientific diagram, taxonomic tree, family tree chart, or text-heavy graphic",
        "An illustration, painting, digital render, or children's book page of a dinosaur",
        "A collage, museum wall layout, or photograph containing multiple different dinosaur skeletons",
        "A close-up macro shot of a single isolated bone, individual tooth, or detached claw"
    ]

    tokenize = clip.tokenize(comparisonprompts).to(device)

    for filename in os.listdir(folderpath):
        filepath = os.path.join(folderpath, filename)
        if not filename.endswith('.jpg'):
            continue

        try:
            image = Image.open(filepath).convert("RGB")
        except Exception:
            os.remove(filepath)
            broken += 1
            continue

        with torch.no_grad():
            imageinput = preprocess(image).unsqueeze(0).to(device)
            logitsperimage, _ = model(imageinput, tokenize)
            probs = logitsperimage.softmax(dim=-1).cpu().numpy()[0]

        # If index 0 is not the highest probability match, trash it
        if probs.argmax() != 0:
            os.remove(filepath)
            irrelevant += 1

    print(f"[{species}] done. Removed {broken} broken, {irrelevant} visually irrelevant/multi-skeleton items.")

# push to kaggle dataset
def kagglepush():
    print("\nprepping upload via kagglehub...")
    try:
        handle = f"{KAGGLE_USERNAME}/{KAGGLE_SLUG}"
        print(f"Pushing to Kaggle dataset handle: {handle}")

        kagglehub.dataset_upload(
            handle=handle,
            local_dataset_dir=datadirectory,
            version_notes="Automated Pipeline Update with Checkpointing & Strict CLIP Filtering"
        )
        print("Upload complete! Check your Kaggle profile.")
    except Exception as e:
        print(f"Kagglehub upload failed: {e}")

# run pipeline with checkpoints
if __name__ == "__main__":
    for dino in specieslist:
        scraping(dino, mIMperSP)
        filter_images(dino)
    kagglepush()

In [ ]:
# CONFIGURATIONS
imagesizing = (224, 224, 3)    # PyTorch image sizing requirement
confidencelevel = (0.85, 0.50) # Confidence thresholds: if confidencelevel, the level of confidence the model has -
                               # in its prediction is 0.85/85% or more, it confirms that exact species. -
                               # Otherwise, graceful fallback is triggered (e.g. Ouranosaurus would simply be -
                               # classified as "unidentified basal hadrosauriform" [or simpler if needbe].)

# safeguard-1: confirms tuples containing image sizing and confidence intervals.
def safeguard1():
  print(imagesizing , confidencelevel)

safeguard1()

# requesting data from the Paleobiology Database's (PBDB) live API; I'm using -
# only accepted species of dinosaur for this project.

import requests
DINO_INFO_url = "https://paleobiodb.org/data1.2/taxa/list.json?base_name=Dinosauria&status=accepted"
def getdinosauria(DINO_INFO_url):
  response = requests.get(DINO_INFO_url)
  dinodata = response.json()
  return dinodata

dinodata = getdinosauria(DINO_INFO_url)
print(dinodata["records"][65])

# parses through data

dinotax2 = {}
for records in dinodata["records"]:
  txnID = records["oid"]
  dinotax2[txnID] = {
      "name": records.get("nam"),
      "rank": records.get("rnk"),
      "parent": records.get("par")
  }

def fallingback(txnID):
  fallbackranks1 = ["family" , "genus" , "superfamily" , "subfamily" , "infraorder"]

  records2 = dinotax2.get(txnID)
  if not records2:
    return "UNKNOWN TAXON" , "UNKNOWN RANK"

  parentID = records2.get("parent")

  while parentID:
    parentrnk = dinotax2.get(parentID)
    if not parentrnk:
      break

    currentrank = parentrnk.get("rank")
    currentname = parentrnk.get("name")

    if currentrank in fallbackranks1:
        return currentname, currentrank

    parentID = parentrnk.get("parent")

  return "Unidentified Dinosauria", "clade"


# safeguard
print(f"{len(dinotax2)} taxons identified")

samplekey1 = list(dinotax2.keys())[67]
print("random taxon:" , dinotax2[samplekey1])

# conditionals concerning previously defined confidence thresholds, classifying -
# specimens, and fallback
highconf = confidencelevel[0]
lowconf = confidencelevel[1]


# how the model guesses
def guessing (species, confscore, txnID):
  if confscore >= highconf:
    print(f"{confscore * 100:.1f}% confident of {species}")
    return species, confscore, None
  elif confscore >= lowconf:
    fallbackname , fallbackrank = fallingback(txnID)
    print(f"{confscore * 100:.1f}% confident of {fallbackname} in {fallbackrank}")
    return fallbackname, confscore, fallbackrank
  else:
    fallbackname , fallbackrank = fallingback(txnID)
    print(f"{confscore * 100:.1f}% confidence, unable to identify; unidentified {fallbackrank} ({fallbackname})")
    return None, confscore, fallbackrank

# another safeguard
sample_key = list(dinotax2.keys())[235]
sample_species_name = dinotax2[sample_key]["name"]
guessing(sample_species_name, 0.28, sample_key)
print("THE ABOVE IS A SAMPLE GUESS!!")

# neural network, image processing

import torch
import torch.nn as neuralnetwork
import torchvision.models as TVmodels
import torchvision.datasets as dataset
from torchvision import transforms
from torch.utils.data import DataLoader

#pytorch standard config, training optimization to avoid model from memorizing training images (aka overfitting)

datatransformation = transforms.Compose([
    transforms.Resize((256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p = 0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])
])

datapath = path
training = dataset.ImageFolder(root = datapath, transform = datatransformation)
speciesamt = len(training.classes)

# another safeguard
print(f"{speciesamt} species found for training")

# data loader
loader = DataLoader(training, batch_size = 50, shuffle  = True)

# resnet model!!

RESNETMODEL = TVmodels.resnet18(weights = TVmodels.ResNet18_Weights.DEFAULT)

# dynamic count
features = RESNETMODEL.fc.in_features
RESNETMODEL.fc = neuralnetwork.Linear(features , speciesamt)
# CUDA
CUDA = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESNETMODEL = RESNETMODEL.to(CUDA)
# loss function and optimization
crit = neuralnetwork.CrossEntropyLoss()
optimizer = torch.optim.AdamW(RESNETMODEL.parameters(), lr = 0.0001, weight_decay = 0.01)
# training loop

epochsnum = 25
# another safeguard
print(f"training on {CUDA}")


for epoch in range(epochsnum):
  RESNETMODEL.train()
  runningloss = 0.0
  correctpredictions = 0
  totalpredictions = 0
  for inputs, labels in loader:
    inputs = inputs.to(CUDA)
    labels = labels.to(CUDA)
    optimizer.zero_grad()
    outputs = RESNETMODEL(inputs)
    loss = crit(outputs, labels)
    loss.backward()
    optimizer.step()

    # batch, summary metrics
    runningloss += loss.item() * labels.size(0)
    _, predicts = torch.max(outputs, dim = 1)
    correctpredictions += torch.sum(predicts == labels).item()
    totalpredictions += labels.size(0)

  epochloss = runningloss / totalpredictions
  epochaccuracy = (correctpredictions / totalpredictions) * 100


# another safeguard, saving
  print(f"Epoch [{epoch+1}/{epochsnum}] - Loss: {epochloss:.4f} - Accuracy: {epochaccuracy:.2f}%")

print("trained")

torch.save(RESNETMODEL.state_dict(), "resnet18DINOSAUR.pth")
print("saved to resnet18DINOSAUR.pth")

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"shayanr10","key":"9d3976c66295ce2c10598814439fdf60"}'}

In [ ]:
import os
import shutil

os.makedirs('/root/.kaggle', exist_ok=True)
if os.path.exists('kaggle.json'):
    shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 600)
    print("New kaggle.json successfully installed!")
else:
    print("Please upload your kaggle.json file using the widget above.")

New kaggle.json successfully installed!
